# This is a notebook

In [134]:
print('Notebook is working')

Notebook is working


In [ ]:
# imports 
from abc import ABC, abstractmethod
from multipledispatch import dispatch
import json
import random


## Map
### Attributes
- Countries
- Continents (maybe)
- Edges

## Country
### Attributes
- Controlling player
- Army size

In [ ]:
class Country:
    """ 
    Risk game country class
     owner is the player that controls the country army_size is the number of 
     armies in the country neighbors is the list of neighboring countries
     (by name)
    """

    def __init__(
            self,
            name: str,
            owner: str,
            army_size: int,
            neighbors: list[Country],
        ):
        self.owner = owner
        self.name = name  
        self.army_size = army_size
        self.neighbors = neighbors

    # getter and setter for owner
    def get_name(self):
        return self.name
    
    def set_name(self, name):
        self.name = name


    def get_owner(self):
        return self.owner   
    
    def set_owner(self, owner):
        self.owner = owner  
   
    # getter and setter for army_size
    def get_army_size(self):
        return self.army_size
    
    def set_army_size(self, army_size):
        self.army_size = army_size
    
    # getter and setter for neighbors
    def get_neighbors(self):    
        return self.neighbors
    
    def set_neighbors(self, neighbors):
        self.neighbors = neighbors    

    #####################################
    # utility function 
    #####################################

    def add_neighbor(self, neighbor):
        """ Adds a neighbor to the country """
        self.neighbors.append(neighbor)

    def is_neighbor(self, country):
        """ Returns true if the country is a neighbor of the country """
        return country in self.neighbors

    def is_controlled_by(self, player):
        """ Returns true if the country is controlled by the player """
        return self.owner == player

    def get_enemy_neighbors(self, player):
        """ Returns the list of enemy neighbors of the country """
        enemy_neighbors = []
        for neighbor in self.neighbors:
            if neighbor.get_owner() != player:
                enemy_neighbors.append(neighbor)
        return enemy_neighbors

    def get_number_of_enemy_neighbors(self, player):
        """ Returns the number of enemy neighbors of the country """
        return len(self.get_enemy_neighbors(player))
    
    def get_friendly_neighbors(self, player):
        """ Returns the list of friendly neighbors of the country """
        friendly_neighbors = []
        for neighbor in self.neighbors:
            if neighbor.get_owner() == player:
                friendly_neighbors.append(neighbor)
        return friendly_neighbors
    
    def get_number_of_friendly_neighbors(self, player):
        """ Returns the number of friendly neighbors of the country """
        return len(self.get_friendly_neighbors(player))


    def __str__(self):
        return self.name      

class Continent:
    """
    Risk game continent class 
        vector of countries and reward for owning the continent (if you control
        all the countries in the continent)
    """

    def __init__(self, countries: list[Country], reward: int):
        self.countries = countries
        self.reward = reward

    # getter and setter for countries
    def get_countries(self):
        return self.countries   
    def set_countries(self, countries):
        self.countries = countries
    
    # getter and setter for reward
    @dispatch()
    def get_reward(self):
        return self.reward
    def set_reward(self, reward):
        self.reward = reward
    
    #####################################
    # utility function 
    #####################################

    def is_controlled_by(self, player):
        """ Returns true if the continent is controlled by the player """
        for country in self.countries:
            if country.get_owner() != player:
                return False
        return True
    
    @dispatch(object)
    def get_reward(self, player):
        """ Returns the reward for controlling the continent """
        if self.is_controlled_by(player):
            return self.reward
        return 0
    
    def get_owned_countries(self, player):
        """ 
            Returns the list of countries in the continent that are controlled
                by the player.
        """
        owned_countries = []
        for country in self.countries:
            if country.get_owner() == player:
                owned_countries.append(country)
        return owned_countries
    

class Map:
    """ Risk game map class """

    def __init__(self, continents: list[Continent]):
        self.continents = continents
    

    # getter and setter for continents
    def get_continents(self):
        return self.continents
    def set_continents(self, continents):
        self.continents = continents

    #####################################
    # utility function 
    #####################################

    def get_reward(self, player):
        """ Returns the reward for controlling the continent """
        reward = 0
        for continent in self.continents:
            reward += continent.get_reward(player)

        reward += self.get_num_countries(player) // 3
        return reward

    # return number of countries controlled by the player
    # maybe this is utilized in the future for calculating the number of armies 
    # to deploy
    def get_num_countries(self, player):
        """ Returns the number of countries controlled by the player """
        num_countries = 0
        for continent in self.continents:
            for country in continent.get_countries():
                if country.get_owner() == player:
                    num_countries += 1
        return num_countries
    


## Actions
- Place armies
- Attack
- Fortify

> Info | notice we assume no presence of the cards in risk


In [ ]:
class Action(ABC):
    """ Action base class """
    @abstractmethod
    def execute(self):
        pass
    
    def __str__(self) -> str:
        return self.__class__.__name__
    
class PlaceArmyAction(Action):
    """ 
        Place army action class 
            - country is the country where the armies will be placed
            - num_armies is the number of armies to place
            
        this is first phase action, the player chooses number of armies to 
            place and the country where to place them, then the action is
            executed and the armies are placed in the country.
    """

    def __init__(self, country: Country, num_armies: int = 0):
        self.country = country
        self.num_armies = num_armies

    def execute(self):
        """ Executes the action """
        self.country.set_army_size(
            self.country.get_army_size() + self.num_armies
        )



class FortifyAction(Action):
    """ Move army action class """
    def __init__(
            self,
            from_country: Country,
            to_country: Country,
            num_armies: int = 0
        ):
        self.from_country = from_country
        self.to_country = to_country
        self.num_armies = num_armies

    def execute(self):
        """ Executes the action """
        self.from_country.set_army_size(
            self.from_country.get_army_size() - self.num_armies)
        self.to_country.set_army_size(
            self.to_country.get_army_size() + self.num_armies)



class MoveAction(FortifyAction):
    """ Fortify action class extend the move army action class, but it is executed in the fortify phase and 
        the player can choose how many armies to move from one country to another
    """
    def __init__(
            self,
            from_country: Country,
            to_country: Country,
            num_armies: int = 0,
            army_want_to_move: int = 0
        ):
        super().__init__(from_country, to_country, num_armies)
        self.army_want_to_move = army_want_to_move

    def execute(self):
        """ Executes the action """
        num_armies_to_move = self.army_want_to_move
        self.from_country.set_army_size(
            self.from_country.get_army_size() - num_armies_to_move)
        self.to_country.set_army_size(
            self.to_country.get_army_size() + num_armies_to_move)



class AttackAction(Action):
    """ 
        Attack action class 
            - from_country is the country from which the attack will be launched
            - to_country is the country that will be attacked
            - num_armies is the number of armies to attack with (must be less 
                         than the number of armies in the from_country)
            
        this is second phase action, the player chooses the country to attack
            from, the country to attack and the number of armies to attack with, 
            then the action is executed and the battle is resolved according to
            the rules of Risk. [see Regolamento.pdf for the rules of Risk] 
    """

    def __init__(
            self,
            from_country: Country,
            to_country: Country,
            num_armies: int = 0,
            army_want_to_move: int = 0,
        ):

        self.army_want_to_move = army_want_to_move
        self.from_country = from_country
        self.to_country = to_country
        self.num_armies = num_armies

    def execute(self):
        """ Executes the action """
        num_attackers = self.num_armies
        
        # notice we assume that the deffenders player always rolls the maximum
        # number of dice (NO CHOOSE THE NUMBER OF DEFFENDERS TANKS)
        # (3 if they have 3 or more armies, 2 if they have 2 armies, 1 if they
        #  have 1 army)
        num_defenders = min(3,self.to_country.get_army_size())

        # classic risk attack rules

        # we roll the dice for the attackers and defenders
        attackers_rolls = sorted([random.randint(1, 6) for _ in range
                                 (num_attackers)] , reverse=True)
        
        defenders_rolls = sorted([random.randint(1, 6) for _ in range
                                 (num_defenders)] , reverse=True)
        
        # we compare the rolls and determine the outcome of the battle
        for attacker_roll, defender_roll in \
            zip(attackers_rolls, defenders_rolls):
            if attacker_roll > defender_roll:
                # attacker wins, defender loses an army
                self.to_country.set_army_size(
                    self.to_country.get_army_size() - 1)
            else:
                # defender wins, attacker loses an army
                self.from_country.set_army_size(
                    self.from_country.get_army_size() - 1)
        
        # we now move the armies from the from_country to the to_country if the attacker
        # has won the battle and conquered the to_country
        if self.to_country.get_army_size() == 0:
            # the attacker has conquered the to_country, we move the armies
                #self.to_country.set_owner(self.from_country.get_owner())
                #self.to_country.set_army_size(num_attackers)
                #self.from_country.set_army_size(
                #    self.from_country.get_army_size() - num_attackers)
            fortify_action = MoveAction(self.from_country, self.to_country, num_attackers, army_want_to_move=self.num_armies)
            fortify_action.execute()

## Game state 

In [ ]:
class GameState:
    """ Game state class """
    def __init__(
            self,
            game_map: Map,
            players: list[str],
            current_player_index: int = 0,
            phase: int = 0
        ):
        self.game_map = game_map
        self.players = players
        self.current_player_index = 0
        self.phase = 0 # 0: place army, 1: attack, 2: fortify

    def get_game_map(self):
        return self.game_map

    def get_current_player(self):
        return self.players[self.current_player_index]
    
    def next_player(self):
        self.current_player_index = \
                             (self.current_player_index + 1) % len(self.players)
    
    def get_phase(self):
        return self.phase

    def set_phase(self, phase):
        self.phase = phase

    def next_phase(self):
        self.phase = (self.phase + 1) % 3
    

## Player (Abstract)

> Notice | A concreate player MUST implement different strategy in `choose_action`

In [ ]:

class Player(ABC):
    """ Player base class """
    
    def __init__(self, color: str):
        self.color = color     

    @abstractmethod
    def choose_action(self, game_state: GameState) -> Action|None:
        """ 
            This method is called during the player's turn to choose an action 
            to execute. The player can choose to execute an action or to end
            their turn (by returning None).
        """
        pass
    
    # TODO: evaluate if we want one MANAGER for chose the action 
    def __str__(self) -> str:
        return self.__class__.__name__
    

    ############################################################################
    #                                                                          #
    #                         !!! NOTICE !!!                                   #    
    #                                                                          #            
    ############################################################################
    # TODO: evaluate where we want put this code here or in the Game class,    #
    # maybe we want to put it in the Game class and call it from the Player    #
    # class but for now we put it here for simplicity                          #
    ############################################################################
    def play_turn(self, game_state : GameState):
        game_state.set_phase(0) # start with place army phase
        self.troops_to_place = game_state.get_game_map().get_reward(self.color)
        while True:
            action = self.choose_action(game_state)
            if action is not None:
                action.execute()
            else:
                game_state.next_phase()
                if game_state.get_phase() == 0:
                    break
    

## Different Player and their strategy

| Player  | Strategy    |
|--------|-----------|
| Red    | Comunist  |
| Purple | Pixie     |
| Yellow | Cluster   |
| Green  | Stinky    |
| Blue   | Neferius  |
| Black  | Angry     |

In [ ]:
################################################################################
# TODO: implement different player classes with different strategies,          #
# for now we will implement only dummy players that                            #
#                                                always end their turn without #
#                                                                              #
################################################################################
# class TreePlayer(Player):                                                    #
#     """ Trees player class """                                               #
#     def choose_action(self, game_state: GameState) -> Action|None:           #
#         """                                                                  #
#             This is a dummy implementation of the choose_action method, it   #
#             always returns None (end turn) but in the future it will be      #
#             implemented with a more sophisticated strategy.                  #
#         """                                                                  #
#         return evaluate(game_state)                                          #
################################################################################

class RedPlayer(Player):
    """ Red player class """
    def choose_action(self, game_state: GameState) -> Action|None:
        """ 
            This is a dummy implementation of the choose_action method, it 
            always returns None (end turn) but in the future it will be 
            implemented with a more sophisticated strategy.
        """
        owned_countries = game_state.get_game_map().get_owned_countries(self.color)


class PurplePlayer(Player):
    """ Purple player class """
    def choose_action(self, game_state: GameState) -> Action|None:
        """ 
            This is a dummy implementation of the choose_action method, it 
            always returns None (end turn) but in the future it will be 
            implemented with a more sophisticated strategy.
        """
        owned_countries = game_state.get_game_map().get_owned_countries(self.color)


class YellowPlayer(Player):
    """ Yellow player class """
    def choose_action(self, game_state: GameState) -> Action|None:
        """ 
            This is a dummy implementation of the choose_action method, it 
            always returns None (end turn) but in the future it will be 
            implemented with a more sophisticated strategy.
        """
        owned_countries = game_state.get_game_map().get_owned_countries(self.color)

    
class GreenPlayer(Player):
    """ Green player class """
    def choose_action(self, game_state: GameState) -> Action|None:
        """ 
            This is a dummy implementation of the choose_action method, it 
            always returns None (end turn) but in the future it will be 
            implemented with a more sophisticated strategy.
        """
        owned_countries = game_state.get_game_map().get_owned_countries(self.color)


class BluePlayer(Player):
    """ Blue player class """
    def choose_action(self, game_state: GameState) -> Action|None:
        """ 
            This is a dummy implementation of the choose_action method, it 
            always returns None (end turn) but in the future it will be 
            implemented with a more sophisticated strategy.
        """
        owned_countries = game_state.get_game_map().get_owned_countries(self.color)


class BlackPlayer(Player):
    """ Black player class """
    def choose_action(self, game_state: GameState) -> Action|None:
        """ 
            Angry is a very simple bot that has attacking as its main strategy.
            
            Angry will place all armies in the country that has the most enemy
            neighbors. 
            
            During the attack phase every country that Angry owns, 
            will attack its weakest neighbor. But it will only attack if the 
            attacking country has more armies than the neighbor. 
            
            If it takes a country all armies will be moved to the country which
            has the most enemy neighbors. The same thinking goes on in the
            fortification phase where any country will give all its armies to 
            any neighbor that has more enemies.
        """
        if game_state.get_phase() == 0:
            # place army phase
            troops = self.troops_to_place 
            if troops > 0:
                owned_countries = game_state.get_game_map().get_owned_countries(self.color)
                get_number_of_enemy_neighbors = lambda country: country.get_number_of_enemy_neighbors(self.color)
                country_to_place = max(owned_countries, key=get_number_of_enemy_neighbors)
                return PlaceArmyAction(country_to_place, troops)
            else:
                return None
        
        elif game_state.get_phase() == 1 :
            # attack phase
            owned_countries = game_state.get_game_map().get_owned_countries(self.color)
            for country in owned_countries:
                enemy_neighbors = country.get_enemy_neighbors(self.color)
                if len(enemy_neighbors) > 0:
                    weakest_enemy_neighbor = min(enemy_neighbors, key=lambda neighbor: neighbor.get_army_size())
                    if country.get_army_size() > weakest_enemy_neighbor.get_army_size() and country.get_army_size() > 1:
                        num_armies_to_attack = country.get_army_size() - 1
                        return AttackAction(country, weakest_enemy_neighbor, num_armies_to_attack) # TODO: WE HAVE TO MOVE AFTER THE ATTACK, CHECK IF THIS MAKE SENSE
            return None

        elif game_state.get_phase() == 2:
            # fortify phase
            owned_countries = game_state.get_game_map().get_owned_countries(self.color)
            for country in owned_countries:
                friendly_neighbors = country.get_friendly_neighbors(self.color)
                for neighbor in friendly_neighbors:
                    if neighbor.get_number_of_enemy_neighbors(self.color) > country.get_number_of_enemy_neighbors(self.color):
                        num_armies_to_fortify = country.get_army_size() - 1
                        if num_armies_to_fortify > 0:
                            return FortifyAction(country, neighbor, num_armies_to_fortify) # TODO: CHECK IF the movment MAKE SENSE
            return None

## Game singleton class
### Attributes
- Map (a non oriented graph)
- Players (Blue, Red, Green, Yellow)
- Current player
- game info

In [141]:
class Game:
    """ Game class of Risk """

    instance = None

    def __init__(self, game_length: int = 100, game_map: Map|None = None):
        self.game_length = game_length # number of turns before the game ends
        self.game_map = game_map
        self.players_list = []
        self.current_player = None
        self.turn = 0
    
    def __new__(cls):
        """ 
         We override the `__new__` method to turn this class into a Singleton
        """
        if cls.instance is None: cls.instance = super().__new__(cls)
        return cls.instance
    
    def set_map(self, game_map: Map):
        """ Sets the game map """
        self.game_map = game_map
    
    def get_map(self) -> Map|None:
        """ Gets the game map """
        return self.game_map
    
    def play():
        """ Plays the game until the end condition is met (game_length turns) """
        while self.turn < self.game_length:
            for player in self.players_list:
                self.current_player = player
                player.choose_action(self.get_game_state()) |> execute 
            self.turn += 1


## Create/Instantiate the map of Risk 

In [142]:
# Definition of the original Risk map with country names, neighbors
# and continent rewards

# North America
alaska = Country("Alaska", "", 0, [])
northwest_territory = Country("Northwest Territory", "", 0, [])
alberta = Country("Alberta", "", 0, [])
ontario = Country("Ontario", "", 0, [])
quebec = Country("Quebec", "", 0, [])
western_united_states = Country("Western United States", "", 0, [])
eastern_united_states = Country("Eastern United States", "", 0, [])
central_america = Country("Central America", "", 0, [])
greenland = Country("Greenland", "", 0, [])

# South America
venezuela = Country("Venezuela", "", 0, [])
peru = Country("Peru", "", 0, [])
brazil = Country("Brazil", "", 0, [])
argentina = Country("Argentina", "", 0, [])

# Europe
iceland = Country("Iceland", "", 0, [])
great_britain = Country("Great Britain", "", 0, [])
scandinavia = Country("Scandinavia", "", 0, [])
northern_europe = Country("Northern Europe", "", 0, [])
western_europe = Country("Western Europe", "", 0, [])
southern_europe = Country("Southern Europe", "", 0, [])
ukraine = Country("Ukraine", "", 0, [])

# Africa
north_africa = Country("North Africa", "", 0, [])
egypt = Country("Egypt", "", 0, [])
east_africa = Country("East Africa", "", 0, [])
madagascar = Country("Madagascar", "", 0, [])
south_africa = Country("South Africa", "", 0, [])
congo = Country("Congo", "", 0, [])

# Asia
ural = Country("Ural", "", 0, [])
siberia = Country("Siberia", "", 0, [])
yakutsk = Country("Yakutsk", "", 0, [])
irkutsk = Country("Irkutsk", "", 0, [])
kamchatka = Country("Kamchatka", "", 0, [])
mongolia = Country("Mongolia", "", 0, [])
japan = Country("Japan", "", 0, [])
afghanistan = Country("Afghanistan", "", 0, [])
china = Country("China", "", 0, [])
india = Country("India", "", 0, [])
siam = Country("Siam", "", 0, [])
middle_east = Country("Middle East", "", 0, [])

# Oceania
indonesia = Country("Indonesia", "", 0, [])
new_guinea = Country("New Guinea", "", 0, [])
western_australia = Country("Western Australia", "", 0, [])
eastern_australia = Country("Eastern Australia", "", 0, [])

# definition of the neighbors for each country

# North America
alaska.set_neighbors([kamchatka, northwest_territory, alberta])
northwest_territory.set_neighbors([alaska, alberta, ontario, greenland])
alberta.set_neighbors([alaska, northwest_territory, ontario,
                       western_united_states])
ontario.set_neighbors([northwest_territory, alberta, western_united_states,
                       eastern_united_states, quebec])
quebec.set_neighbors([ontario, eastern_united_states, greenland])
western_united_states.set_neighbors([alberta, ontario, eastern_united_states,
                                     central_america])
eastern_united_states.set_neighbors([ontario, quebec, western_united_states,
                                     central_america])
central_america.set_neighbors([western_united_states, eastern_united_states,
                               venezuela])
greenland.set_neighbors([northwest_territory, quebec, iceland])

# South America
venezuela.set_neighbors([central_america, peru, brazil])
peru.set_neighbors([venezuela, brazil, argentina])
brazil.set_neighbors([venezuela, peru, argentina, north_africa])
argentina.set_neighbors([peru, brazil])

# Europe
iceland.set_neighbors([greenland, great_britain, scandinavia])
great_britain.set_neighbors([iceland, scandinavia, northern_europe,
                             western_europe])
scandinavia.set_neighbors([iceland, great_britain, northern_europe, ukraine])
northern_europe.set_neighbors([great_britain, scandinavia, western_europe,
                               southern_europe, ukraine])
western_europe.set_neighbors([great_britain, northern_europe, southern_europe,
                              north_africa])
southern_europe.set_neighbors([northern_europe, western_europe, ukraine, egypt,
                               north_africa])
ukraine.set_neighbors([scandinavia, northern_europe, southern_europe, ural,
                       afghanistan, middle_east])

# Africa
north_africa.set_neighbors([western_europe, southern_europe, egypt,
                            east_africa, brazil, congo])
egypt.set_neighbors([southern_europe, north_africa, east_africa,
                     middle_east])
east_africa.set_neighbors([north_africa, egypt, madagascar, south_africa,
                           middle_east, congo])
madagascar.set_neighbors([east_africa, south_africa])
south_africa.set_neighbors([east_africa, madagascar, congo])
congo.set_neighbors([north_africa, east_africa, south_africa])

# Asia
ural.set_neighbors([ukraine, siberia, china, afghanistan])
siberia.set_neighbors([ural, yakutsk, irkutsk, mongolia, china])
yakutsk.set_neighbors([siberia, irkutsk, kamchatka])
irkutsk.set_neighbors([siberia, yakutsk, mongolia, kamchatka])
kamchatka.set_neighbors([alaska, yakutsk, irkutsk, mongolia, japan])
mongolia.set_neighbors([siberia, irkutsk, kamchatka, japan, china])
japan.set_neighbors([kamchatka, mongolia])
afghanistan.set_neighbors([ukraine, ural, china, india, middle_east])
china.set_neighbors([ural, siberia, mongolia, afghanistan, india, siam])
india.set_neighbors([afghanistan, china, siam, middle_east])
siam.set_neighbors([china, india, indonesia])
middle_east.set_neighbors([ukraine, southern_europe, egypt, east_africa,
                           afghanistan, india])

# Oceania
indonesia.set_neighbors([siam, new_guinea, western_australia])
new_guinea.set_neighbors([indonesia, western_australia, eastern_australia])
western_australia.set_neighbors([indonesia, new_guinea, eastern_australia])
eastern_australia.set_neighbors([new_guinea, western_australia])

# Creation of continents with their countries and rewards

# North America - reward: 5
north_america = Continent(
    [alaska, northwest_territory, alberta, ontario, quebec,
     western_united_states, eastern_united_states, central_america, greenland],
    5
)

# South America - reward: 2
south_america = Continent(
    [venezuela, peru, brazil, argentina],
    2
)

# Europe - reward: 5
europe = Continent(
    [iceland, great_britain, scandinavia, northern_europe,
     western_europe, southern_europe, ukraine],
    5
)

# Africa - reward: 3
africa = Continent(
    [north_africa, egypt, east_africa, madagascar, south_africa, congo],
    3
)

# Asia - reward: 7
asia = Continent(
    [ural, siberia, yakutsk, irkutsk, kamchatka, mongolia, japan,
     afghanistan, china, india, siam, middle_east],
    7
)

# Australia/Oceania - reward: 2
australia = Continent(
    [indonesia, new_guinea, western_australia, eastern_australia],
    2
)

# Creation of the complete Risk map
risk_map = Map([north_america, south_america, europe, africa, asia, australia])

# Example usage to print the map
print("=== ORIGINAL RISK MAP ===")
print(f"Continents: {len(risk_map.get_continents())}")
print(f"Total countries: {risk_map.get_num_countries('')}\n")

# Quick check of the number of countries per continent
print("\n=== COUNTRY DISTRIBUTION PER CONTINENT ===")
for continent in risk_map.get_continents():
    print(f"{continent.__class__.__name__}: "
          f"{len(continent.get_countries())} countries")

=== ORIGINAL RISK MAP ===
Continents: 6
Total countries: 42


=== COUNTRY DISTRIBUTION PER CONTINENT ===
Continent: 9 countries
Continent: 4 countries
Continent: 7 countries
Continent: 6 countries
Continent: 12 countries
Continent: 4 countries
